# GP10 — Clip / Intersection / Overlay: Ενδεικτική λύση

Χρησιμοποιούμε clip για περιορισμό έκτασης και overlay intersection για συνδυασμό polygon attributes. Σε αντίθεση με spatial join, οι πράξεις αυτές δημιουργούν derived geometries. Το solution κρατά χωριστά τα clipped και overlay layers για traceability.

In [ ]:
import geopandas as gpd
from pathlib import Path
root = Path('data/gp10')
study = gpd.read_file(root / 'study_area.geojson').to_crs('EPSG:2100')
landuse = gpd.read_file(root / 'landuse.geojson').to_crs('EPSG:2100')
protected = gpd.read_file(root / 'protected_zones.geojson').to_crs('EPSG:2100')


In [ ]:
clipped = gpd.clip(landuse, study)
overlay = gpd.overlay(clipped, protected, how='intersection')
clipped['area_m2'] = clipped.geometry.area
overlay['area_m2'] = overlay.geometry.area
print('landuse before/after:', len(landuse), len(clipped))
print('overlay rows:', len(overlay))


In [ ]:
for frame in (study, clipped, overlay):
    assert frame.crs.to_epsg() == 2100
    assert frame.geometry.notna().all()
    assert frame.geometry.is_empty.sum() == 0
    assert frame.geometry.is_valid.all()
    assert frame.total_bounds[0] >= study.total_bounds[0] - 1
assert clipped.geom_type.isin(['Polygon','MultiPolygon']).all()
assert overlay.geom_type.isin(['Polygon','MultiPolygon']).all()
assert (clipped['area_m2'] > 0).all()
assert (overlay['area_m2'] > 0).all()
assert overlay['area_m2'].sum() <= clipped['area_m2'].sum()


In [ ]:
clipped.to_file('gp10_results.gpkg', layer='clipped_landuse', driver='GPKG')
overlay.to_file('gp10_results.gpkg', layer='landuse_protected', driver='GPKG')
print('exported clip and overlay layers')
